# 0077: the description reader on real prompts, arm R

0074's reader (arm L: the split, a folded description after each check, free values) learned instructions of 2 to 8
parts, all worded from our templates. Arm R is the same recipe with real prompts in the mix: 2,000 rows whose task is a
person's own prompt (OASST1, English and French), with 2 to 8 checks put before it, after it or after its first
sentence. It is read beside arm L on prompts neither saw (OASST1 and Dolly-15k).

The set is `rules/view_real_sft_r.jsonl`: 10,000 train records (0074's 8,000 and the 2,000 real-prompt ones) and 300
held (dev_real). No Dolly text and no test row is in it. About 1,560 steps.

**Runtime:** an A100 or the G4. It reads, from Drive: the base, the tokenizer and the set; it writes
`rules/cubby450m_view_real_r/` (`talk_lora.*`). Training only: the read is its own notebook.
**Disconnect when it prints `trained:`.**

In [ ]:
import hashlib, json, os, signal, subprocess, sys, time
BRANCH = 'master'
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q -b {BRANCH} https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git fetch -q && git checkout -q {BRANCH} && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'

DATA = f'{DRIVE}/rules/view_real_sft_r.jsonl'
OUT  = f'{DRIVE}/rules/cubby450m_view_real_r'
for p in (CKPT, TOKENIZER, DATA, DATA + '.manifest.json'):
    assert os.path.exists(p), p
man = json.load(open(DATA + '.manifest.json'))
print(man['records'], 'records', man['by_split'], man['by_family'])
assert man['order'] == '0077' and man['arm'] == 'm', "this is not order 0077's set in arm M's target"
assert man['by_split'] == {'train': 10000, 'held': 300}, man['by_split']
assert man['dolly records'] == 0 and man['rows of test_real or test_dolly'] == 0 and man['rows of test or test_long'] == 0
assert man['answers that read back strictly, also with own_text'] == man['records'], 'answers the grammar would refuse'
assert man['records_over_max_len'] == {}, f"records longer than MAX_LEN would be skipped: {man['records_over_max_len']}"
assert {'IFEval', 'IFBench'} <= set(man['benchmarks_checked']), 'the benchmark check did not run'
assert man["0074's train records, the same as 0074's set"] is True, "0074's records were not checked against its set"
rows = json.load(open(f'{REPO}/validation/logs/long_rows_0077_2026-11-18_manifest.json'))['sha256']
assert man['rows_sha256']['long_rows_0077'] == rows, "the set's rows are not the rows whose hashes are committed: do not train"
print('rows as committed:', {s: h[:12] for s, h in rows['the lines of rows.jsonl by set'].items()})
!nvidia-smi --query-gpu=name,memory.total --format=csv

## The settings

0074's, unchanged: rank 16, alpha 32 (every projection of every layer), lr 2e-4 with 50 warmup steps then cosine,
batch 32, max length 1,280, five epochs, seed 0, key tokens weighted 4x. On a card under 70 GB a batch is cut into
chunks of at most 4,096 padded tokens (`--micro-tokens`: the same gradient).

In [ ]:
EPOCHS, BATCH, LR, RANK, ALPHA, MAX_LEN, KEY_WEIGHT = 5, 32, 2e-4, 16, 32, 1280, 4.0
os.environ['CB_NO_FLEX'] = '1'                # the reference attention path; the trainer does not compile flex
assert not os.path.exists(f'{OUT}/DONE'), f'{OUT} is already trained: move it aside to train again'
os.makedirs(OUT, exist_ok=True)
tag = '_view_real_r'
import torch
MICRO = 0 if torch.cuda.get_device_properties(0).total_memory >= 70e9 else 4096
print('micro-batch tokens:', MICRO or 'off (the whole batch at once)')
cmd = [sys.executable, '-u', 'validation/train_talk_torch.py', '--ckpt', CKPT, '--tokenizer', TOKENIZER,
       '--data', DATA, '--out', OUT, '--epochs', str(EPOCHS), '--batch', str(BATCH), '--lr', str(LR),
       '--rank', str(RANK), '--alpha', str(ALPHA), '--max-len', str(MAX_LEN), '--seed', '0', '--tag', tag,
       '--micro-tokens', str(MICRO), '--key-weight', str(KEY_WEIGHT)]
p = None
try:
    with open(f'{OUT}/train.log', 'a', encoding='utf-8', buffering=1) as f:
        f.write(f"\n=== {time.strftime('%F %T')} {' '.join(cmd[2:])}\n")
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=''); f.write(line)
        assert p.wait() == 0, f'the trainer failed ({p.returncode})'
finally:
    if p and p.poll() is None:
        p.send_signal(signal.SIGINT)
subprocess.run(f'cp {REPO}/validation/logs/train_talk_torch{tag}.* {OUT}/ 2>/dev/null', shell=True)
json.dump({'prompts': 'OASST1 (Apache-2.0)', 'templates': "this repo's own (Apache-2.0)"},
          open(f'{OUT}/row_licences.json', 'w'))
open(f'{OUT}/DONE', 'w').write(time.strftime('%F %T') + '\n')
print('trained:', OUT)

## Done

**Disconnect the runtime.** Next: `view_real_read_dev.ipynb` (arm R and arm L on dev_real, for looking).